# Lab 5\. The Complete Model Lifecycle: Train, Evaluate, and Use a Rock-Paper-Scissors Classifier

## Overview

In the Chapter 5 lessons you walked through each stage of the PyTorch model lifecycle on its own: you built a small network by subclassing **nn.Module**, ran a four-step training loop, measured a trained model with accuracy and a confusion matrix, and saved, reloaded, and used a model to predict. This lab is where you run all six stages as one connected pipeline on the real Rock-Paper-Scissors image data, from loading the images through to a saved model that predicts the class of a new picture.

You will build the **RPSClassifier** model, place it and the data on the best available device, train it while watching the loss fall, evaluate it on a held-out test set, and read the gap between training and test performance. You will then reach a guided decision point where you change the number of training epochs and watch how training loss and test accuracy respond, which turns the idea of "training longer" into a number you can measure. Finally you will save the trained model to a file, reload it into a fresh model, and run a single-image prediction, which completes both the lifecycle and the classification problem the course set out in Chapter 2\.

The workflow you assemble here is the same one you will use for every model in later courses. A junior machine learning engineer asked to reproduce or extend a model starts exactly this way: load the data, build the network, train, check the numbers on unseen data, and save the result so it can be reused. Running the whole sequence yourself, and seeing how a choice in one stage shows up in a later one, is the practical skill this course was built to give you.

## Learning Objectives

By the end of this lab, you should be able to:

* Build a simple neural network classifier by subclassing **nn.Module** and place it on the active device.

* Write and run a training loop that performs the forward pass, loss computation, backward pass, and parameter update over several epochs.

* Evaluate a trained model on a test set by computing accuracy and a confusion matrix, and interpret the train-test gap as overfitting.

* Measure how the number of training epochs affects training loss and test accuracy by running a controlled experiment.

* Save a trained model's state dictionary to a file, reload it into a fresh model instance, and run a prediction on a new image.

## Prerequisites

This course’s labs are **not cumulative**, so this lab does not depend on any prior resources or the state of earlier labs if there are any.

You need the following before you start:

* **A notebook environment:** either a Google account for Google Colab, or a local installation of Jupyter Notebook. All steps in this lab run on a CPU. A CUDA GPU (in Colab, enabled through the runtime settings) or an Apple Silicon (MPS) device makes training faster but is not required.

* **Python 3.12 or later.** On Colab this is already provided. For a local environment, confirm your version with **python \--version**.

* **PyTorch 2.12.1 and TorchVision 0.27,** installed with **pip** in the first setup step below. If you are on Colab, these may already be present; the setup step confirms the versions.

* **matplotlib**, used once to display a sample image. It is pre-installed on Colab and in most local scientific-Python setups.

* **A web browser** (Chrome, Firefox, or Edge are recommended for Colab).


## Estimated Time

Approximately 60-75 minutes. The part that varies most is training the model a few times, including the epochs experiment in Step 4\. On a CPU each training run of ten epochs takes a few minutes; on a GPU it is faster.

## How to Perform the Exercise

You will work entirely in one notebook, running cells from top to bottom. Create a new notebook in Google Colab (**File \> New notebook**) or a new **.ipynb** in local Jupyter. Download the chapter 5 lab notebook from the [LFD2000-resources GitHub repository](https://github.com/lftraining/LFD2000-resources.git) from the **LFD2000-resources/labs** directory. In Google Colab, select File, then Upload Notebook, and select the chapter 5 lab notebook you downloaded.

Read the explanation before running each cell, run it, and compare what you see to the **Sample Output** or **Output** shown after each block. Sample Output means the exact values will differ from run to run (random tensors, dataset counts, file paths), so match the shape and structure rather than every character. Output marks a checkpoint where a specific outcome confirms the step worked.

The rock-paper-scissors images come from the [LFD2000-resources GitHub repository](https://github.com/lftraining/LFD2000-resources.git), which you clone in Setup 3\. That repository holds the dataset under **LFD2000-resources/datasets/RPS/train**, organized as one folder per class (rock, paper, and scissors). You do not modify any files in the dataset; you only read the images and turn them into tensors.

Commands in this lab use the notebook form, where a line beginning with **\!** runs a shell command from inside a notebook cell (for example, **\!pip install ...**). If you run these in a local terminal instead of a notebook cell, drop the leading **\!**.

> **Note:** Run the cells in order and only once each, unless a step says otherwise. If you restart the notebook's runtime, re-run the cells from the top, because the variables live in memory and are cleared on restart.

## Setup: Environment Preparation

The steps in this section only prepare the environment. They are not the learning content of the lab, and on a managed course platform some of them may already be done for you. Run them once at the top of your notebook, then move on to Step 1\.

### Setup 1\. Install PyTorch and TorchVision, and Import

Install the two libraries the lab uses. PyTorch provides tensors and the operations on them; TorchVision provides the image dataset loader and the image transforms used at the end of the lab.You need PyTorch for tensors and device operations, and TorchVision for loading images as tensors. Install both, then import the pieces the lab uses. Installing in the first cell makes the notebook self-contained, so anyone who opens it later gets the right libraries.


In [ ]:
!pip install torch torchvision


Sample Output

```
Successfully installed torch-2.12.1 torchvision-0.27.0
```

Confirm the versions so your results line up with this lab. Printing the versions now also verifies the import works before you depend on it later.


In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)


Sample Output

```
PyTorch: 2.12.1
TorchVision: 0.27.0
```

If your versions are close to these but not identical, the lab still works; the tensor and transform APIs used here are stable across the 2.x line.

### Setup 2\. Import Libraries and Select the Device

Before any model code, you import the pieces you will use and choose the device once, so every tensor and model can be routed to the same place. The imports cover the core library (**torch**), the neural-network module (**torch.nn**), the **DataLoader** that feeds data in batches, and the TorchVision dataset and transform tools you used in Chapter 3\.


In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.transforms import v2

# Device-agnostic selection, carried from Chapter 4
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Using device:", device)


The device block is the exact pattern from Chapter 4: it checks for a CUDA GPU first, then Apple Silicon through MPS, and falls back to the CPU when neither is present. Storing the choice in **device** lets you send the model and every batch to the same hardware later.

Sample Output

```
Using device: cpu
```

### Setup 3\. Get the Data, Split It, and Build the DataLoaders

The model learns from the Rock-Paper-Scissors images, so you download them, load them into a training set and a test set, and wrap each set so it can be fed to the model in batches.

First, get the data. The images live in the LFD2000 repository, which you clone once. In a notebook, a line beginning with **\!** runs a shell command:


In [ ]:
# Run once to download the dataset (notebook shell command)
!git clone https://github.com/LinuxFoundation/LFD2000-resources.git


This creates an **LFD2000-resources** directory containing both **datasets/RPS/train** and **datasets/RPS/test** which hold one subfolder per class.

There is one tool in this step that was not used in the Chapter 3 lessons, so here is what it does. A **DataLoader** wraps a dataset and hands it to the training loop in small batches, and it can shuffle the data each epoch; the training loop in Step 2 expects exactly this, an object it can loop over to get batches of images and labels.


In [ ]:
# The Chapter 3 image-to-tensor pipeline: resize, convert to a float tensor, scale to [0, 1]
to_tensor = v2.Compose([
    v2.ToImage(),
    v2.Resize((128, 128)),
    v2.ToDtype(torch.float32, scale=True),
])

# Load the whole dataset from the one-folder-per-class layout
train_dataset = ImageFolder("../datasets/RPS/train", transform=to_tensor)
test_dataset = ImageFolder("../datasets/RPS/test", transform=to_tensor)

# Read the class names from the ImageFolder, in label-index order
class_names = train_dataset.classes
print("Classes and their label indices:", train_dataset.class_to_idx)

# Feed the data to the model in shuffled batches during training
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Training images: {len(train_dataset)}, test images: {len(test_dataset)}")


The **to\_tensor** pipeline is the same one from Chapter 3, so the images arrive in the shape the model expects. Loading **LFD2000-resources/datasets/RPS/train** with **ImageFolder** gives you one dataset holding every labeled image in the training set.  The test set is loaded from a different folder, **LFD2000-resources/datasets/RPS/test,** and it stays unseen during training because the training loop only ever reads **train\_loader**. Each **DataLoader** groups its set into batches of 32; you shuffle the training data so the model does not see the images in the same order every epoch, and you leave the test data unshuffled because order does not affect a test score.

Reading the class names matters for turning predictions into words later. **ImageFolder** decides the integer label for each class by sorting the folder names alphabetically, so the mapping is not necessarily the order you would guess. You read the names from the **classes** attribute of the **ImageFolder** object. Reading from the **ImageFolder** gives **class\_names\[0\]** as the name of label **0**, whatever the folders are called, and you use this list in Step 6 to turn a numeric prediction into a readable answer.

Sample Output

```
Classes and their label indices: {'paper': 0, 'rock': 1, 'scissors': 2}
Training images: 2520, test images: 372
```

> **Note:** Because **ImageFolder** orders labels alphabetically, the mapping above has **paper** as label 0, not **rock**. This is why you read **class\_names** from the **classes** attribute in **ImageFolder** rather than from the list of class folders; writing the list by hand in a different order would leave the model training correctly but every printed prediction mislabeled.

## Step 1\. Build the Model

With the data ready, you build the network. This is the build stage of the lifecycle. The model is the **RPSClassifier** from the chapter: it flattens each image into a long vector, passes it through a linear layer and a ReLU activation, and ends with a linear layer that produces three numbers, one score per class.

You define the model as a class that inherits from **nn.Module** and fill in two methods: **\_\_init\_\_**, where you create the layers, and **forward**, where you say how an input tensor flows through them. The first linear layer needs to know how many numbers are in one flattened image. Rather than write that number by hand, you read it from a sample image with **.numel()**, which counts the values in the tensor. This keeps the model correct even if the image size behind the transform changes.


In [ ]:
class RPSClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_classes=3):
        super().__init__()               # required: sets up the nn.Module machinery
        self.flatten = nn.Flatten()      # (C, H, W) image -> one flat vector
        self.fc1 = nn.Linear(input_size, hidden_size)   # first linear layer
        self.relu = nn.ReLU()            # non-linear activation
        self.fc2 = nn.Linear(hidden_size, num_classes)  # output: one score per class

    def forward(self, x):
        x = self.flatten(x)   # flatten each image in the batch
        x = self.fc1(x)       # first linear layer
        x = self.relu(x)      # activation
        logits = self.fc2(x)  # final scores, one per class
        return logits

# Read the flattened input size from one sample image
sample_image, _ = train_dataset[0]
input_size = sample_image.numel()   # numbers in one flattened image

torch.manual_seed(52)			# initializes the model using a seed
model = RPSClassifier(input_size=input_size, num_classes=len(class_names)).to(device)
print(model)


The call to **super().\_\_init\_\_()** runs the setup **nn.Module** needs and always comes first. Taking **sample\_image** from **train_dataset\[0\]** gives you one image tensor, and **.numel()** returns its total number of values, which is the flattened input size the first linear layer expects. You create the model with that input size and **num\_classes** set from the length of **class\_names**, then move the whole model to the device with **.to(device)**. There is one additional line that we need to discuss: setting the manual seed. Seeds are important for reproducibility since they ensure the model is randomly initialized in the same way every time the code runs. Printing the model shows its structure so you can confirm the layers are wired the way you expect.

Sample Output

```
RPSClassifier(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=49152, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=3, bias=True)
)
```

The **in\_features** value depends on the image size. For a **(3, 128, 128\)** image it is **3 \* 128 \* 128 \= 49152**; if the resize in Setup uses a different size, this number changes to match, which is the point of reading it from the data.

## Step 2\. Train the Model

A freshly built model has random internal numbers, so its predictions are no better than guessing. Training adjusts those numbers until the model's answers line up with the correct labels. This is the train stage, and it is the core of the lab. You set up two tools, then run a loop.

### Step 2.1. Set Up the Loss Function and Optimizer

Training needs a loss function to measure how wrong the model is and an optimizer to adjust the model's numbers. You use **nn.CrossEntropyLoss** for classification, which takes the raw logits and the correct labels and returns one loss number per batch. You use **torch.optim.Adam** as the optimizer, created with the model's parameters and a learning rate.


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # lr is the learning rate


**model.parameters()** hands the optimizer every learnable number in the model. The learning rate, **lr=1e-3**, controls how large each adjustment is. These two choices are reasonable defaults for this problem; the chapter treats the question of why they work, and what alternatives exist, as a topic for later study.

### Step 2.2. Run the Training Loop

Training happens one batch at a time, and for each batch the same four steps run in order: the forward pass produces logits, the loss function scores them against the labels, the backward pass computes gradients, and the optimizer updates the parameters. The inner loop runs those four steps over every batch; the outer loop repeats the whole pass for several epochs, where one epoch is a full pass over the training data.

You put the model into training mode with **model.train()** before the loop. When you run the loop, the printed loss should trend downward from epoch to epoch, which is the visible sign that the model is learning.


In [ ]:
num_epochs = 10
model.train()   # put the model in training mode

for epoch in range(num_epochs):
    running_loss = 0.0
    for images, labels in train_loader:          # inner loop: one batch at a time
        images = images.to(device)               # keep data on the model's device
        labels = labels.to(device)

        logits = model(images)                    # 1. forward pass
        loss = loss_fn(logits, labels)            # 2. loss computation

        optimizer.zero_grad()                     # clear old gradients
        loss.backward()                           # 3. backward pass (autograd)
        optimizer.step()                          # 4. parameter update

        running_loss += loss.item()               # .item() gets the plain number

    average_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch + 1}: loss = {average_loss:.4f}")


Two lines inside the loop are worth naming. Moving **images** and **labels** to **device** on every batch is what keeps the data on the same hardware as the model, which prevents the device mismatch error from Chapter 4\. The **optimizer.zero\_grad()** call clears the previous batch's gradients before the backward pass computes new ones, because PyTorch adds gradients together by default (that is called gradient accumulation); skipping it makes each batch's gradients pile onto the last and training goes wrong in ways that are hard to spot.

Sample Output

```
Epoch 1: loss = 3.0212
Epoch 2: loss = 0.5257
Epoch 3: loss = 0.4496
...
Epoch 10: loss = 0.0994
```

A falling loss tells you training is working. It does not tell you whether the model can classify images it has never seen, which is what you measure next.

## Step 3\. Evaluate the Model

Evaluation measures the trained model on the test images it never saw during training. This is the evaluate stage. You switch the model to evaluation mode, turn off gradient tracking, and compute two things: overall accuracy, and a confusion matrix that shows which classes the model confuses.

### Step 3.1. Measure Accuracy on the Test Set

You call **model.eval()** to put the model in evaluation mode, and you wrap the loop in **torch.inference\_mode()** to turn off gradient tracking, because you are only reading answers, not training. For each test batch you run the images through the model, take the **argmax** of the logits to get the predicted class, and count how many predictions match the labels.


In [ ]:
model.eval()   # evaluation mode
correct = 0
total = 0

with torch.inference_mode():                     # no gradient tracking
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        predictions = logits.argmax(dim=1)       # class with the highest score
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print(f"Test accuracy: {accuracy:.2%}")


**model.eval()** and **torch.inference\_mode()** do two different jobs and are used together: the first sets the behavior of the layers, and the second turns off gradient tracking. The expression **logits.argmax(dim=1)** returns, for each image, the index of the highest score, which is the predicted class. Comparing **predictions \== labels** gives a tensor of True and False values, and **.sum().item()** counts the True ones as a plain number.

Sample Output

```
Test accuracy: 52.96%
```

### Step 3.2. Build a Confusion Matrix and Read Precision and Recall

Accuracy gives one number for the whole test set, which can hide uneven performance: a model might handle rock and paper well and confuse scissors often, yet still post a decent overall score. A confusion matrix shows the details. It is a table that counts, for every true class, how many test images were predicted as each class. The diagonal holds the correct predictions, and the off-diagonal cells show which classes get mistaken for which.

You can build the confusion matrix with the tensor tools from Chapter 3, without any extra library. You create a square table of zeros, then add one to the cell at row "true label" and column "predicted label" for every test image.


In [ ]:
num_classes = len(class_names)
confusion = torch.zeros(num_classes, num_classes, dtype=torch.int64)

model.eval()
with torch.inference_mode():
    for images, labels in test_loader:
        images = images.to(device)
        logits = model(images)
        predictions = logits.argmax(dim=1).cpu()   # bring predictions back to CPU

        for true_label, predicted_label in zip(labels, predictions):
            confusion[true_label, predicted_label] += 1

print("Confusion matrix (rows = true class, columns = predicted class):")
print(confusion)

# Per-class precision and recall, derived from the confusion matrix
correct_per_class = confusion.diag()
precision = correct_per_class / confusion.sum(dim=0)   # column sums = predicted totals
recall = correct_per_class / confusion.sum(dim=1)      # row sums = actual totals

for i, name in enumerate(class_names):
    print(f"{name:>10}: precision = {precision[i]:.2f}, recall = {recall[i]:.2f}")


The predictions are moved back to the CPU with **.cpu()** so they can index the **confusion** table, which lives on the CPU. After the loop, the diagonal of the table (**confusion.diag()**) holds the count of correct predictions for each class. Precision for a class is the correct count divided by how many images the model assigned to that class, which is the column sum. Recall for a class is the correct count divided by how many images truly belong to that class, which is the row sum.

Sample Output

```
Confusion matrix (rows = true class, columns = predicted class):
tensor([[38,   46,  40],
        [  31,  58,  35],
        [ 3,  20,  101]])
     paper: precision = 0.53, recall = 0.31
      rock: precision = 0.47, recall = 0.47
  scissors: precision = 0.57, recall = 0.81
```

Reading the matrix row by row tells you where the model struggles. In the sample above, most values sit on the diagonal, and the off-diagonal counts show the handful of images each class loses to the other two. Your exact numbers will differ, and the class order follows **class\_names**.

### Step 3.3. Interpret the Result: Overfitting

For this course, the expected result is that the model does well on the training data and noticeably worse on the test data. That gap has a name: overfitting, which is when a model learns the training examples so closely that it fails to generalize to new ones. It happens here because the model is trained without the techniques that hold overfitting back.

This outcome is a deliberate part of the exercise. Seeing the gap yourself is the clearest motivation for the techniques that reduce it, such as regularization and data augmentation, which are the subject of later courses. If you want to see the training-side number for comparison, you can run the same accuracy loop over **train\_loader** instead of **test\_loader**; the training accuracy will typically be higher than the test accuracy you measured in Step 3.1.

## Step 4\. Experiment: Vary the Number of Epochs

This is the guided decision point. You have trained for ten epochs and measured one test accuracy. Now you change the number of epochs and watch how the test accuracy responds, which shows how training length affects what the model learns.

One detail makes this a fair comparison: each trial must start from a freshly built model. If you keep training the same model, the epochs add up and you cannot tell what a given epoch count produces on its own. To keep each trial independent, you wrap building, training, and evaluating in a single function, then call it with different epoch counts.


In [ ]:
torch.manual_seed(52)

def train_and_evaluate(num_epochs):
    # Build a fresh model so each trial starts from the same random state
    fresh_model = RPSClassifier(input_size=input_size, num_classes=len(class_names)).to(device)
    optimizer = torch.optim.Adam(fresh_model.parameters(), lr=1e-3)

    fresh_model.train()
    for epoch in range(num_epochs):
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = fresh_model(images)
            loss = loss_fn(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Measure test accuracy for this epoch count
    fresh_model.eval()
    correct = 0
    total = 0
    with torch.inference_mode():
        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)
            predictions = fresh_model(images).argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return correct / total


for epochs_to_try in [1, 5, 10, 20]:
    accuracy = train_and_evaluate(epochs_to_try)
    print(f"{epochs_to_try:>2} epochs -> test accuracy = {accuracy:.2%}")


The function rebuilds the model and its optimizer every time it is called, trains for the requested number of epochs, and returns the test accuracy. The loop then tries a short run, a medium run, and two longer runs, printing the accuracy for each. Running this trains the model four separate times, so expect it to take a little longer than a single run.

Sample Output

```
 1 epochs -> test accuracy = 48.66%
 5 epochs -> test accuracy = 50.81%
10 epochs -> test accuracy = 56.99%
20 epochs -> test accuracy = 58.87%
```

Look at how the numbers move rather than at any single value. Test accuracy usually climbs quickly over the first several epochs, then flattens, and it may even dip as training continues, because the model keeps fitting the training data past the point where that helps on unseen images. This is overfitting shown as a curve: more training does not keep improving the test score. Your exact numbers will differ from the sample, but the shape of the trend is the thing to observe.

### Step 4.1 (Optional) Experiment \- Change the image size

What do you think will happen if the model were trained on smaller images? Would it perform better or worse?  
Go back to “Setup 3” and change the second transformation (Resize) to v2.Resize((28, 28)) and run the whole pipeline once again, recreating a model suitable for the new image size, and training it.

Once you’re finished, compare losses, accuracy, and confusion matrices of both models. Did they change in the way you expected them to change? If not, could you think of a reason behind these results?

## Step 5\. Save the Trained Model

Training a model takes time, and a model lives only in memory: when the notebook restarts, the trained numbers are gone. The save stage writes those numbers to a file so you can reuse them. The recommended approach in PyTorch is to save the model's state dictionary, which maps each layer to its learned weights and biases, rather than the whole model object.

You save the **model** you trained in Step 2\. The experiment in Step 4 built and trained separate models inside the **train\_and\_evaluate** function and never changed **model**, so **model** still holds the weights it learned in Step 2\. Save its state dictionary with **torch.save**.


In [ ]:
torch.save(model.state_dict(), "rps_model.pth")
print("Saved model to rps_model.pth")


**model.state\_dict()** returns the dictionary of learned numbers, and **torch.save** writes it to the file **rps\_model.pth**. The **.pth** extension is a common convention for PyTorch model files. Saving the state dictionary rather than the whole model keeps the file independent of your exact code layout, which makes it load reliably later.

**Output:** a file named **rps\_model.pth** appears in your working directory, and the confirmation line prints.

```
Saved model to rps_model.pth
```

> **Note:** Saving the state dictionary saves only the learned numbers, not the shape of the network. To load them back you need the **RPSClassifier** class defined in your notebook so you can build an empty model of the same shape first. Keep that class in place; the file alone is not enough to rebuild the model.

## Step 6\. Reload the Model and Predict on a New Image

The last two stages complete the lifecycle. You reload the saved numbers into a fresh model, then use that model to predict the class of a new image.

### Step 6.1. Reload Into a Fresh Model

Loading reverses the save in three steps: build a new model instance with the same architecture, read the saved dictionary from disk with **torch.load**, and load those numbers into the new model with **load\_state\_dict**. Because you are loading the model to make predictions, you finish by switching it to evaluation mode.


In [ ]:
loaded_model = RPSClassifier(input_size=input_size, num_classes=len(class_names)).to(device)
loaded_model.load_state_dict(torch.load("rps_model.pth", weights_only=True))
loaded_model.eval()
print("Model reloaded and set to evaluation mode")


The **weights\_only=True** argument tells **torch.load** to read only the saved numbers and not to run any other code stored in the file, which is the safer default. Building the new model on **device** first keeps it consistent with the images you will feed it.

Output

```
Model reloaded and set to evaluation mode
```

> **Warning:** **load\_state\_dict** takes the dictionary that **torch.load** returns, not a file path. Writing **loaded\_model.load\_state\_dict("rps\_model.pth")** passes a string where a dictionary is expected and fails. Load the file first, then pass the result, as shown above.

### Step 6.2. Predict on a New Image

Prediction, also called inference, uses the trained model to get an answer for new data. You take one image from the test set as a stand-in for a new picture, add a batch dimension of size one with **unsqueeze(0)** because the model expects a batch even for a single image, run it through the model inside an inference context, take the **argmax** to get the predicted class index, and map that index to a readable name using **class\_names**.


In [ ]:
# Use one test image as a stand-in for a new picture
new_image, true_label = test_dataset[0]
new_image = new_image.to(device)

with torch.inference_mode():
    logits = loaded_model(new_image.unsqueeze(0))   # add a batch dimension of 1
    predicted_index = logits.argmax(dim=1).item()

print("Predicted:", class_names[predicted_index])
print("Actual:   ", class_names[true_label])


**new\_image.unsqueeze(0)** turns the single image of shape **(3, 128, 128\)** into a batch of one, shape **(1, 3, 128, 128\)**, which is what the model expects. The **argmax** picks the class with the highest score, and **.item()** pulls that index out as a plain Python number. Because you read **class\_names** from the dataset in Setup, the printed name matches the label the model was trained on.

Sample Output

```
Predicted: paper
Actual:    paper
```

The prediction may not always match the actual label, since the model is not perfect, and that is expected given the overfitting gap you measured. With this step, the model reads an image and returns rock, paper, or scissors, which is the goal the course set out in Chapter 2\.

## Cleanup

This is the final lab of Course 1, so nothing here needs to persist for a later chapter. Cleanup removes the two on-disk artifacts the lab created: the saved model file **rps\_model.pth** and the cloned **LFD2000-resources** dataset directory.

On Google Colab, the runtime is temporary. When you disconnect or the session ends, the installed packages and the cloned **LFD2000-resources** folder are cleared automatically, and nothing else remains. No action is required. If you want to keep the trained model, download **rps\_model.pth** to your own machine before the session ends.

On a local Jupyter environment, the artifacts on disk are the saved model file and the cloned dataset folder. You can remove it if you want to return to your pre-lab state, or keep it to save re-downloading.


In [ ]:
import os

# Remove the saved model file
if os.path.exists("rps_model.pth"):
    os.remove("rps_model.pth")
    print("Removed rps_model.pth")
else:
    print("No model file to remove")

# Remove the cloned dataset directory (notebook shell command)
!rm -rf LFD2000-resources
print("Removed the LFD2000-resources directory")


Output

```
Removed rps_model.pth
Removed the LFD2000 directory
```

This cleanup is complete: removing **rps\_model.pth** and the **LFD2000-resources** directory returns the working environment to its pre-lab state Keep either artifact only if you plan to rerun or extend the lab.

## Summary

In this lab you assembled the full PyTorch model lifecycle end to end. You loaded the Rock-Paper-Scissors images and batched them with a **DataLoader**, built the **RPSClassifier** and placed it on the active device, trained it with the four-step loop while watching the loss fall, and evaluated it on unseen test data with accuracy, a confusion matrix, precision, and recall. You saw the train-test gap that signals overfitting, and in the epochs experiment you watched test accuracy climb and then flatten as training length grew, which turns overfitting from a definition into a trend you produced yourself. You then saved the trained model, reloaded it into a fresh instance, and used it to predict the class of a new image, completing both the lifecycle and the classification problem the course introduced in Chapter 2\. The same six stages, in the same order, are what every model you build in later courses will follow.
